# 🚀 SpaceX Falcon 9 – Notebook 7: Plotly Dash Dashboard

> **Run this as a Python script**, not a notebook cell (or use the separate `spacex_dash_app.py` file).

In [ ]:
!pip install dash plotly pandas --quiet
print("Dash installed!")

In [ ]:
# Save the Dash app to a .py file, then run: python spacex_dash_app.py
dash_code = '''
import pandas as pd
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px

df = pd.read_csv('dataset_part_2.csv')
max_payload = df['PayloadMass'].max()
min_payload = df['PayloadMass'].min()

app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1('SpaceX Launch Records Dashboard',
            style={'textAlign':'center','color':'#503D36','font-size':'40px'}),

    dcc.Dropdown(
        id='site-dropdown',
        options=[{'label':'All Sites','value':'ALL'}] +
                [{'label':s,'value':s} for s in df['LaunchSite'].unique()],
        value='ALL',
        placeholder='Select a Launch Site',
        searchable=True
    ),

    html.Br(),
    dcc.Graph(id='success-pie-chart'),
    html.Br(),

    html.P("Payload range (kg):"),
    dcc.RangeSlider(
        id='payload-slider',
        min=0, max=10000, step=1000,
        marks={i: str(i) for i in range(0,10001,2500)},
        value=[min_payload, max_payload]
    ),
    dcc.Graph(id='success-payload-scatter-chart'),
])

@app.callback(
    Output('success-pie-chart','figure'),
    Input('site-dropdown','value')
)
def update_pie(selected_site):
    if selected_site == 'ALL':
        fig = px.pie(df, values='Class', names='LaunchSite',
                     title='Total Success Launches by Site')
    else:
        filtered = df[df['LaunchSite'] == selected_site]
        fig = px.pie(filtered, names='Class',
                     title=f'Success vs Failure: {selected_site}')
    return fig

@app.callback(
    Output('success-payload-scatter-chart','figure'),
    [Input('site-dropdown','value'), Input('payload-slider','value')]
)
def update_scatter(selected_site, payload_range):
    low, high = payload_range
    filtered = df[(df['PayloadMass'] >= low) & (df['PayloadMass'] <= high)]
    if selected_site != 'ALL':
        filtered = filtered[filtered['LaunchSite'] == selected_site]
    fig = px.scatter(filtered, x='PayloadMass', y='Class',
                     color='BoosterVersion',
                     title='Payload vs. Launch Outcome')
    return fig

if __name__ == '__main__':
    app.run(debug=True)
'''

with open('spacex_dash_app.py', 'w') as f:
    f.write(dash_code)
print("spacex_dash_app.py saved! Run it with: python spacex_dash_app.py")